# Transformation Testing
=======================

Manual testing notebook for data transformations from src/transformations.py

In [ ]:
# Import transformation functions
import sys
sys.path.insert(0, '..')

from src.transformations import (
    calculate_risk_bucket,
    calculate_claim_status_category,
    calculate_vehicle_category,
    apply_risk_buckets,
    apply_claim_status_categories,
    apply_vehicle_categories,
    Config,
    RISK_BUCKET_EXCELLENT,
    RISK_BUCKET_GOOD,
    RISK_BUCKET_FAIR,
    RISK_BUCKET_POOR,
    RISK_BUCKET_UNKNOWN,
    CLAIM_STATUS_OPEN,
    CLAIM_STATUS_CLOSED,
    CLAIM_STATUS_DENIED,
    CLAIM_STATUS_OTHER,
    VEHICLE_CATEGORY_CAR,
    VEHICLE_CATEGORY_SUV,
    VEHICLE_CATEGORY_TRUCK,
    VEHICLE_CATEGORY_MOTORCYCLE,
    VEHICLE_CATEGORY_OTHER,
)

print("Imported transformation functions:")
print(f"  - calculate_risk_bucket")
print(f"  - calculate_claim_status_category")
print(f"  - calculate_vehicle_category")
print(f"  - apply_risk_buckets")
print(f"  - apply_claim_status_categories")
print(f"  - apply_vehicle_categories")

## Risk Bucket Tests

In [ ]:
# Test risk bucket calculation
print("=== Risk Bucket Tests ===\n")

test_scores = [
    (850, RISK_BUCKET_EXCELLENT),
    (800, RISK_BUCKET_EXCELLENT),
    (750, RISK_BUCKET_EXCELLENT),
    (749, RISK_BUCKET_GOOD),
    (700, RISK_BUCKET_GOOD),
    (699, RISK_BUCKET_FAIR),
    (650, RISK_BUCKET_FAIR),
    (649, RISK_BUCKET_POOR),
    (300, RISK_BUCKET_POOR),
    (None, RISK_BUCKET_UNKNOWN),
]

passed = 0
for score, expected in test_scores:
    result = calculate_risk_bucket(score)
    status = "✓" if result == expected else "✗"
    if result != expected:
        status = f"✗ FAIL: score={score} expected={expected} got={result}"
        print(status)
    else:
        passed += 1
        print(f"✓ score={score} → {result}")

print(f"\n{passed}/{len(test_scores)} tests passed")

## Claim Status Tests

In [ ]:
# Test claim status category
print("=== Claim Status Category Tests ===\n")

test_statuses = [
    ("Open", CLAIM_STATUS_OPEN),
    ("Closed", CLAIM_STATUS_CLOSED),
    ("Denied", CLAIM_STATUS_DENIED),
    ("open", CLAIM_STATUS_OTHER),  # case sensitive
    ("Pending", CLAIM_STATUS_OTHER),
    ("Reopened", CLAIM_STATUS_OTHER),
    ("", CLAIM_STATUS_OTHER),
    (None, CLAIM_STATUS_OTHER),
]

passed = 0
for status, expected in test_statuses:
    result = calculate_claim_status_category(status)
    if result == expected:
        passed += 1
        print(f"✓ status='{status}' → {result}")
    else:
        print(f"✗ FAIL: status='{status}' expected={expected} got={result}")

print(f"\n{passed}/{len(test_statuses)} tests passed")

## Vehicle Category Tests

In [ ]:
# Test vehicle category
print("=== Vehicle Category Tests ===\n")

test_vehicles = [
    ("Sedan", VEHICLE_CATEGORY_CAR),
    ("Coupe", VEHICLE_CATEGORY_CAR),
    ("Wagon", VEHICLE_CATEGORY_CAR),
    ("SUV", VEHICLE_CATEGORY_SUV),
    ("Truck", VEHICLE_CATEGORY_TRUCK),
    ("Motorcycle", VEHICLE_CATEGORY_MOTORCYCLE),
    ("Van", VEHICLE_CATEGORY_OTHER),
    ("Coupe", VEHICLE_CATEGORY_CAR),
    (None, VEHICLE_CATEGORY_OTHER),
    ("", VEHICLE_CATEGORY_OTHER),
]

passed = 0
for vehicle, expected in test_vehicles:
    result = calculate_vehicle_category(vehicle)
    if result == expected:
        passed += 1
        print(f"✓ vehicle='{vehicle}' → {result}")
    else:
        print(f"✗ FAIL: vehicle='{vehicle}' expected={expected} got={result}")

print(f"\n{passed}/{len(test_vehicles)} tests passed")

## DataFrame Transformation Tests

In [ ]:
import pandas as pd

# Test apply_risk_buckets on DataFrame
print("=== DataFrame: apply_risk_buckets ===\n")

df = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005"],
    "credit_score": [800, 720, 650, 300, None],
})

result = apply_risk_buckets(df)
print(result)
print(f"\nRisk bucket distribution:")
print(result["risk_bucket"].value_counts())

In [ ]:
# Test apply_claim_status_categories on DataFrame
print("=== DataFrame: apply_claim_status_categories ===\n")

df = pd.DataFrame({
    "claim_id": ["CLM001", "CLM002", "CLM003", "CLM004"],
    "claim_status": ["Open", "Closed", "Denied", "Pending"],
})

result = apply_claim_status_categories(df)
print(result)
print(f"\nClaim status category distribution:")
print(result["claim_status_category"].value_counts())

In [ ]:
# Test apply_vehicle_categories on DataFrame
print("=== DataFrame: apply_vehicle_categories ===\n")

df = pd.DataFrame({
    "claim_id": ["CLM001", "CLM002", "CLM003", "CLM004", "CLM005"],
    "vehicle_type": ["Sedan", "SUV", "Truck", "Motorcycle", "Van"],
})

result = apply_vehicle_categories(df)
print(result)
print(f"\nVehicle category distribution:")
print(result["vehicle_category"].value_counts())

## Load Real Data and Apply Transformations

In [ ]:
# Load real data from MinIO and apply transformations
import os
import boto3

MINIO_CONFIG = {
    "endpoint": os.getenv("MINIO_ENDPOINT", "localhost:9900"),
    "access_key": os.getenv("MINIO_ROOT_USER", "minioadmin"),
    "secret_key": os.getenv("MINIO_ROOT_PASSWORD", "minioadmin"),
    "bucket": os.getenv("MINIO_BUCKET", "insurance-data"),
}

s3 = boto3.client(
    "s3",
    endpoint_url=f"http://{MINIO_CONFIG['endpoint']}",
    aws_access_key_id=MINIO_CONFIG["access_key"],
    aws_secret_access_key=MINIO_CONFIG["secret_key"],
)

def list_objects(prefix: str) -> list:
    response = s3.list_objects_v2(Bucket=MINIO_CONFIG["bucket"], Prefix=prefix)
    return [obj["Key"] for obj in response.get("Contents", [])]

def read_parquet(key: str) -> pd.DataFrame:
    local_path = f"/tmp/{os.path.basename(key)}"
    s3.download_file(MINIO_CONFIG["bucket"], key, local_path)
    df = pd.read_parquet(local_path)
    os.unlink(local_path)
    return df

# Find silver customers
silver_customer_keys = [k for k in list_objects("silver/silver_customers/") if k.endswith(".parquet")]

if silver_customer_keys:
    df_silver = read_parquet(silver_customer_keys[0])
    print(f"Loaded silver customers: {len(df_silver)} rows")
    print("\nColumns:", list(df_silver.columns))
    print("\nSample:")
    print(df_silver.head())
    
    if "risk_bucket" in df_silver.columns:
        print("\nRisk bucket distribution:")
        print(df_silver["risk_bucket"].value_counts())
else:
    print("No silver customer data found")

In [ ]:
# Find silver claims
silver_claim_keys = [k for k in list_objects("silver/silver_claims/") if k.endswith(".parquet")]

if silver_claim_keys:
    df_silver = read_parquet(silver_claim_keys[0])
    print(f"Loaded silver claims: {len(df_silver)} rows")
    print("\nColumns:", list(df_silver.columns))
    print("\nSample:")
    print(df_silver.head())
    
    if "claim_status_category" in df_silver.columns:
        print("\nClaim status category distribution:")
        print(df_silver["claim_status_category"].value_counts())
    
    if "vehicle_category" in df_silver.columns:
        print("\nVehicle category distribution:")
        print(df_silver["vehicle_category"].value_counts())
else:
    print("No silver claim data found")

## Configuration Check

In [ ]:
# Check configuration
print("=== Configuration ===\n")

print("PostgreSQL:")
print(f"  Host: {Config.POSTGRES_HOST}:{Config.POSTGRES_PORT}")
print(f"  Database: {Config.POSTGRES_DB}")
print(f"  User: {Config.POSTGRES_USER}")

print("\nMinIO:")
print(f"  Endpoint: {Config.MINIO_ENDPOINT}")
print(f"  Bucket: {Config.MINIO_BUCKET}")

print("\nClickHouse:")
print(f"  Host: {Config.CLICKHOUSE_HOST}:{Config.CLICKHOUSE_PORT}")
print(f"  Database: {Config.CLICKHOUSE_DB}")
print(f"  User: {Config.CLICKHOUSE_USER}")